In [5]:
import os
os.chdir('/ictstr01/home/icb/fatemehs.hashemig/codes/interpretable-ssl')

In [6]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
from importlib import reload

import interpretable_ssl.trainers.scproto_utils
import interpretable_ssl.datasets.dataset_configs
import interpretable_ssl.datasets.dataset
import interpretable_ssl.configs.defaults
import interpretable_ssl.evaluation.de_helper
import interpretable_ssl.evaluation.cd4_marker
import interpretable_ssl.evaluation.metric_helpers.embedding_metrics
import interpretable_ssl.augmenters.graph_generator
import interpretable_ssl.augmenters.adata_augmenter
import interpretable_ssl.models.swav
import interpretable_ssl.trainers.base
import interpretable_ssl.trainers.trainer
import interpretable_ssl.trainers.adaptive_trainer
import interpretable_ssl.trainers.scproto

reload(interpretable_ssl.trainers.scproto_utils)
reload(interpretable_ssl.datasets.dataset_configs)
reload(interpretable_ssl.datasets.dataset)
reload(interpretable_ssl.configs.defaults)
reload(interpretable_ssl.evaluation.de_helper)
reload(interpretable_ssl.evaluation.cd4_marker)
reload(interpretable_ssl.evaluation.metric_helpers.embedding_metrics)
reload(interpretable_ssl.augmenters.graph_generator)
reload(interpretable_ssl.augmenters.adata_augmenter)
reload(interpretable_ssl.models.swav)
reload(interpretable_ssl.trainers.base)
reload(interpretable_ssl.trainers.trainer)
reload(interpretable_ssl.trainers.adaptive_trainer)
reload(interpretable_ssl.trainers.scproto)
from interpretable_ssl.trainers.scproto import *

from interpretable_ssl.evaluation.metric_helpers.embedding_tables import *


# t = SCProtoTrainer(debug=1, workers=0, dataset_id = '0.3pancreas', full_dataset_mode = 1, affinity_type = 'arbf')
# t.setup()

 captum (see https://github.com/pytorch/captum).
INFO:faiss.loader:Loading faiss with AVX2 support.
INFO:faiss.loader:Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
INFO:faiss.loader:Loading faiss.
INFO:faiss.loader:Successfully loaded faiss.


In [7]:
t = SCProtoTrainer(debug=1, workers=0, dataset_id = 's28nsc', affinity_type = 'c2', lambda_kl = 0.0, lambda_proto_recon = 0.5, p = 0.0, version=25, experiment_name= 'swav')

t.setup()


INFO:root:['NVIDIA A100 80GB PCIe']


dataset is None, loading s28nsc
loading s28nsc data
⚠️ No HVG column found.
Embedding dictionary:
 	Num conditions: [1]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 960 31 10
	Mean/Var Layer in/out: 31 8
Decoder Architecture:
	First Layer in, out and cond:  8 31 10
	Output Layer in/out:  31 960 



INFO:root:=======>Building model done. max value for adata fed to scpoli_wrapper: 8.688319206237793


adam


INFO:root:Optimizer 'adam' built successfully.


In [8]:
import numpy as np
import scipy.sparse as sp

def affinity_report(A: sp.csr_matrix):
    A = A.tocsr(copy=True)
    A.setdiag(0)
    A.eliminate_zeros()

    nnz = np.diff(A.indptr)
    mean_deg = nnz.mean()
    med_deg = np.median(nnz)

    # row entropy + effective neighbors
    ent = np.zeros(A.shape[0])
    effk = np.zeros(A.shape[0])
    for i in range(A.shape[0]):
        s, e = A.indptr[i], A.indptr[i+1]
        if s == e: 
            ent[i] = 0.0
            effk[i] = 0.0
            continue
        p = A.data[s:e]
        p = p / (p.sum() + 1e-12)
        ent[i] = -(p * np.log(p + 1e-12)).sum()
        effk[i] = np.exp(ent[i])

    # mutual edges ratio
    B = A.astype(bool)
    mutual = (B.multiply(B.T)).sum()
    total = B.sum()
    mutual_ratio = float(mutual / (total + 1e-12))

    return {
        "mean_deg": float(mean_deg),
        "med_deg": float(med_deg),
        "effk_mean": float(effk.mean()),
        "effk_med": float(np.median(effk)),
        "mutual_ratio": mutual_ratio,
        "frac_empty_rows": float((nnz == 0).mean()),
    }


In [9]:
affinity_report(t.train_ds.aff)

{'mean_deg': 8.000034233093132,
 'med_deg': 8.0,
 'effk_mean': 7.883870922797854,
 'effk_med': 7.923713047629706,
 'mutual_ratio': 0.44516951727266113,
 'frac_empty_rows': 0.0}

In [10]:
def load_model(name, t):
    model = t.get_model()
    checkpoint_path = f'{t.get_save_dir()}/{name}/checkpoint.pth.tar'
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint["state_dict"])
    model.to(t.device)
    return model

# task 1

Embedding dictionary:
 	Num conditions: [1]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 960 31 10
	Mean/Var Layer in/out: 31 8
Decoder Architecture:
	First Layer in, out and cond:  8 31 10
	Output Layer in/out:  31 960 



# task 2

In [280]:
# scib, scgraph

In [281]:
# rbo

In [282]:
# batch mask recovery

# task 3

In [283]:
# higher purity for rare cell types

In [284]:
# better f1 classification based on mc label